# 04 holiday 实验

## 本文件目标
验证粗粒度 holiday 特征是否能在 safe lag 基础上继续带来增益。

## 本文件主要工作
1. 分析 holidays_events 表
2. 说明为什么不能直接按 date merge 原始节日表
3. 构造日级节日特征：
   - is_holiday
   - holiday_count
4. 合并到 safe lag 版本主表中
5. 重新训练并比较结果

## 当前限制
这版 holiday 特征是粗粒度的：
- 没区分 National / Regional / Local
- 没结合具体门店 city/state
- 没区分不同商品家族对节日的敏感程度

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor

PATH = "/kaggle/input/competitions/store-sales-time-series-forecasting/"

train = pd.read_csv(PATH + "train.csv", parse_dates=["date"])
test = pd.read_csv(PATH + "test.csv", parse_dates=["date"])
stores = pd.read_csv(PATH + "stores.csv")
oil = pd.read_csv(PATH + "oil.csv", parse_dates=["date"])
holidays = pd.read_csv(PATH + "holidays_events.csv", parse_dates=["date"])
transactions = pd.read_csv(PATH + "transactions.csv", parse_dates=["date"])
sample_submission = pd.read_csv(PATH + "sample_submission.csv")

In [ ]:
base_train = train.merge(stores, on="store_nbr", how="left")
base_test = test.merge(stores, on="store_nbr", how="left")

base_train = base_train.merge(oil, on="date", how="left")
base_test = base_test.merge(oil, on="date", how="left")

base_train = base_train.sort_values("date").copy()
base_test = base_test.sort_values("date").copy()

base_train["dcoilwtico"] = base_train["dcoilwtico"].ffill().bfill()
base_test["dcoilwtico"] = base_test["dcoilwtico"].ffill().bfill()

In [ ]:
train_df = base_train.copy()
test_df = base_test.copy()

for df in [train_df, test_df]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["dayofyear"] = df["date"].dt.dayofyear
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

cat_cols = ["store_nbr", "family", "city", "state", "type", "cluster"]
num_cols = ["onpromotion", "dcoilwtico", "year", "month", "day", "dayofweek", "dayofyear", "is_weekend"]
feature_cols = cat_cols + num_cols
target_col = "sales"

for col in cat_cols:
    train_df[col] = train_df[col].astype(str)
    test_df[col] = test_df[col].astype(str)

In [ ]:
lag_df = pd.concat([
    train_df[["id", "date", "store_nbr", "family", "sales"]].copy(),
    test_df[["id", "date", "store_nbr", "family"]].assign(sales=np.nan).copy()
], ignore_index=True)

lag_df = lag_df.sort_values(["store_nbr", "family", "date"]).copy()

lag_df["lag_16"] = lag_df.groupby(["store_nbr", "family"])["sales"].shift(16)
lag_df["lag_21"] = lag_df.groupby(["store_nbr", "family"])["sales"].shift(21)
lag_df["lag_28"] = lag_df.groupby(["store_nbr", "family"])["sales"].shift(28)

train_df_lag = train_df.merge(
    lag_df[["id", "lag_16", "lag_21", "lag_28"]],
    on="id",
    how="left"
)

test_df_lag = test_df.merge(
    lag_df[["id", "lag_16", "lag_21", "lag_28"]],
    on="id",
    how="left"
)

for col in ["lag_16", "lag_21", "lag_28"]:
    train_df_lag[col] = train_df_lag[col].fillna(0)
    test_df_lag[col] = test_df_lag[col].fillna(0)

feature_cols_lag = feature_cols + ["lag_16", "lag_21", "lag_28"]

## 为什么不能直接 merge 原始节日表
`holidays_events.csv` 同一天可能有多条记录，直接按 `date` merge 会炸表。所以这里先按日期聚合成一张日级节日特征表。

In [ ]:
hol = holidays.copy()
hol = hol[hol["transferred"] == False].copy()

hol["is_holiday"] = hol["type"].isin(
    ["Holiday", "Additional", "Bridge", "Transfer"]
).astype(int)

holiday_features = hol.groupby("date").agg(
    is_holiday=("is_holiday", "max"),
    holiday_count=("description", "count")
).reset_index()

train_df_hol = train_df_lag.merge(
    holiday_features,
    on="date",
    how="left"
)

test_df_hol = test_df_lag.merge(
    holiday_features,
    on="date",
    how="left"
)

train_df_hol["is_holiday"] = train_df_hol["is_holiday"].fillna(0)
train_df_hol["holiday_count"] = train_df_hol["holiday_count"].fillna(0)
test_df_hol["is_holiday"] = test_df_hol["is_holiday"].fillna(0)
test_df_hol["holiday_count"] = test_df_hol["holiday_count"].fillna(0)

feature_cols_hol = feature_cols_lag + ["is_holiday", "holiday_count"]

In [ ]:
cutoff_date = train_df_hol["date"].max() - pd.Timedelta(days=365)
train_df_small = train_df_hol[train_df_hol["date"] >= cutoff_date].copy()

last_date = train_df_small["date"].max()
val_start_date = last_date - pd.Timedelta(days=15)

train_part = train_df_small[train_df_small["date"] < val_start_date].copy()
valid_part = train_df_small[train_df_small["date"] >= val_start_date].copy()

X_train = train_part[feature_cols_hol].copy()
X_valid = valid_part[feature_cols_hol].copy()
X_test = test_df_hol[feature_cols_hol].copy()

y_train = np.log1p(train_part[target_col].values)
y_valid = np.log1p(valid_part[target_col].values)

model = CatBoostRegressor(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=50
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid),
    use_best_model=True,
    early_stopping_rounds=50
)

In [ ]:
def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

valid_pred_log = model.predict(X_valid)
valid_pred = np.expm1(valid_pred_log)
valid_pred = np.clip(valid_pred, 0, None)

valid_true = valid_part[target_col].values
score = rmsle(valid_true, valid_pred)
print("Validation RMSLE:", score)

## 当前结果
- experiment_name: `safe_lag_plus_coarse_holiday_catboost_recent365`
- features_used: `stores + oil + basic_date_features + lag_16 + lag_21 + lag_28 + is_holiday + holiday_count`
- validation_split: `last_16_days`
- model: `CatBoostRegressor`
- params: `iterations=300, depth=6, lr=0.05`
- score: `0.460751`

## 当前结论
粗粒度 holiday 特征已成功加入模型，但效果不如 safe lag 版本。

- safe lag: 0.447398
- safe lag + coarse holiday: 0.460751

因此当前不保留粗粒度 holiday 作为主版本。